# RAG como Tool de Agente

No notebook anterior, montamos um RAG que responde perguntas sobre documentos internos. Mas ele só faz isso: sempre busca nos documentos, mesmo que a pergunta não tenha nada a ver com eles.

Neste notebook, vamos transformar o RAG em uma **tool** que o agente pode usar quando quiser. O agente passa a ter duas fontes de informação:
- **Base interna** (RAG): para perguntas sobre produtos, políticas e procedimentos da empresa
- **Busca na web**: para informações externas e atualizadas

O agente decide autonomamente qual fonte consultar com base na pergunta.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Preparando a base de conhecimento

Vamos repetir o setup do RAG de forma compacta: carregar, dividir e indexar os documentos.

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

loader = TextLoader("resources/base_conhecimento.md")
documentos = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(documentos)

vectorstore = InMemoryVectorStore.from_documents(
    chunks,
    OpenAIEmbeddings(model="text-embedding-3-small")
)

print(f"Base de conhecimento pronta: {len(chunks)} chunks indexados.")

Base de conhecimento pronta: 7 chunks indexados.


## Criando a tool de RAG

A tool encapsula a busca no vector store. O agente chama essa tool quando precisa de informações internas. A descrição da tool é fundamental: é o que o modelo lê para decidir quando usá-la.

In [3]:
from langchain.tools import tool

@tool
def consultar_base_conhecimento(pergunta: str) -> str:
    """Consulta a base de conhecimento interna da NovaTech Financeira.
    Use para perguntas sobre produtos, taxas, politicas, procedimentos e servicos da empresa.
    NAO use para informacoes externas ou de mercado."""
    resultados = vectorstore.similarity_search(pergunta, k=3)
    contexto = "\n\n".join([doc.page_content for doc in resultados])
    return f"Informacoes encontradas na base interna:\n\n{contexto}"

In [4]:
print(consultar_base_conhecimento.invoke("taxas do cartao"))

Informacoes encontradas na base interna:

Para abrir uma conta, o cliente precisa ser maior de 18 anos, possuir CPF válido e realizar a verificação de identidade pelo aplicativo (selfie + documento).

### Cartão de Crédito

O Cartão de Crédito NovaTech possui três modalidades:

**NovaTech Essencial:**
- Sem anuidade
- Limite inicial de R$ 500 a R$ 5.000
- Cashback de 0,5% em todas as compras
- Parcelamento em até 12x sem juros em lojas parceiras

**NovaTech Plus:**
- Anuidade de R$ 19,90/mês (isenta para gastos acima de R$ 3.000/mês)
- Limite de R$ 5.000 a R$ 25.000
- Cashback de 1,0% em todas as compras
- Sala VIP em aeroportos nacionais (2 acessos por ano)
- Parcelamento em até 18x sem juros em lojas parceiras

**NovaTech Black:**
- Anuidade de R$ 59,90/mês (isenta para gastos acima de R$ 8.000/mês)
- Limite de R$ 25.000 a R$ 100.000
- Cashback de 1,5% em todas as compras
- Sala VIP ilimitada em aeroportos nacionais e internacionais
- Seguro viagem internacional incluso
- Concierge 2

## Tool de busca na web

A segunda fonte de informação do agente: busca na web via Tavily para informações externas e atualizadas.

In [5]:
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def buscar_na_web(query: str) -> Dict[str, Any]:
    """Busca informacoes atualizadas na internet.
    Use para informacoes externas, noticias, cotacoes e dados que nao sao da empresa."""
    return tavily_client.search(query)

## Agente com RAG + Web

Agora criamos o agente com as duas tools. O system prompt orienta quando usar cada uma.

In [6]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agente = create_agent(
    model="gpt-4.1-mini",
    tools=[consultar_base_conhecimento, buscar_na_web],
    system_prompt=(
        "Voce e um assistente da NovaTech Financeira. "
        "Para perguntas sobre produtos, taxas, politicas e procedimentos da NovaTech, "
        "consulte a base de conhecimento interna. "
        "Para informacoes externas, noticias ou dados de mercado, use a busca na web. "
        "Sempre indique de onde veio a informacao."
    )
)

## Pergunta interna

Uma pergunta sobre os produtos da empresa deve acionar a base de conhecimento.

In [7]:
resposta = agente.invoke(
    {"messages": [HumanMessage(content="Quais sao os tipos de cartao de credito da NovaTech e suas anuidades?")]}
)

print(resposta["messages"][-1].content)

A NovaTech oferece três tipos de cartão de crédito, com as seguintes anuidades:

1. NovaTech Essencial
   - Sem anuidade
   - Limite inicial de R$ 500 a R$ 5.000
   - Cashback de 0,5% em todas as compras
   - Parcelamento em até 12x sem juros em lojas parceiras

2. NovaTech Plus
   - Anuidade de R$ 19,90 por mês (isenta para gastos acima de R$ 3.000/mês)
   - Limite de R$ 5.000 a R$ 25.000
   - Cashback de 1,0% em todas as compras
   - Sala VIP em aeroportos nacionais (2 acessos por ano)
   - Parcelamento em até 18x sem juros em lojas parceiras

3. NovaTech Black
   - Anuidade de R$ 59,90 por mês (isenta para gastos acima de R$ 8.000/mês)
   - Limite de R$ 25.000 a R$ 100.000
   - Cashback de 1,5% em todas as compras
   - Sala VIP ilimitada em aeroportos nacionais e internacionais
   - Seguro viagem internacional incluso
   - Concierge 24 horas

Se quiser mais detalhes sobre algum cartão, posso ajudar!


## Pergunta externa

Uma pergunta sobre o mercado deve acionar a busca na web.

In [8]:
resposta = agente.invoke(
    {"messages": [HumanMessage(content="Qual e a cotacao do dolar hoje?")]}
)

print(resposta["messages"][-1].content)

A cotação do dólar hoje está em torno de R$ 5,22 a R$ 5,23, com algumas variações um pouco abaixo ou acima desse valor dependendo da fonte. Por exemplo, o dólar comercial está cotado aproximadamente a R$ 5,22 e o dólar turismo em torno de R$ 5,42. Essas informações são atualizadas em tempo real e podem variar durante o dia. (Fonte: dolarhoje.com, dolarhoje.net.br)


## Pergunta mista

A verdadeira prova de fogo: uma pergunta que exige **ambas as fontes** — dados internos da empresa e informações externas de mercado.

In [9]:
resposta = agente.invoke(
    {"messages": [HumanMessage(content="Quais fundos de investimento a NovaTech oferece? E como esta o Ibovespa hoje?")]}
)

print(resposta["messages"][-1].content)

A NovaTech oferece os seguintes fundos de investimento:

- NovaTech Conservador: fundo de renda fixa, com taxa de administração de 0,3% ao ano.
- NovaTech Moderado: fundo multimercado, com taxa de administração de 0,8% ao ano.
- NovaTech Arrojado: fundo de ações, com taxa de administração de 1,2% ao ano.

Além disso, a NovaTech oferece opções de investimentos em CDB, LCI, LCA, ações, fundos imobiliários (FIIs) e ETFs, tudo acompanhado em tempo real pelo aplicativo, sem taxa de custódia.

Sobre o Ibovespa hoje, o índice está variando em torno de 182.335 pontos, tendo subido aproximadamente 1,37% no dia conforme dados recentes do mercado (hora aproximada 10:47 AM GMT-3). (Fonte: Investing.com, TradingView, Yahoo Finance). 

Se desejar, posso ajudar com mais detalhes sobre os fundos ou dados de mercado.


## Inspecionando as decisões

Podemos ver quais tools o agente chamou em cada resposta. Na pergunta mista, ele deve ter usado as duas.

In [10]:
for msg in resposta["messages"]:
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"Tool chamada: {tc['name']}")
            print(f"  Argumentos: {tc['args']}")
            print()

Tool chamada: consultar_base_conhecimento
  Argumentos: {'pergunta': 'Quais fundos de investimento a NovaTech oferece?'}

Tool chamada: buscar_na_web
  Argumentos: {'query': 'Ibovespa today'}



O agente usou `consultar_base_conhecimento` para os fundos da NovaTech e `buscar_na_web` para o Ibovespa. Cada parte da pergunta foi roteada para a fonte correta de forma autônoma.

Esse padrão de **RAG como tool** é o mais flexível: o agente tem acesso ao conhecimento interno sem perder a capacidade de buscar informações externas. A descrição da tool é o que guia essa decisão, por isso ela precisa ser clara e específica.